# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VenkataVishnuVardhanReddy/Flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**Plain English Rule Definition:**
"A page is prioritized for an editorial refresh if it receives significant organic visibility (impressions >= 500), ranks outside the safe top 3 spots (average position > 3.0) where it faces high ranking volatility, and exhibits a below-average click-through rate (CTR < 1.5%), which suggests search intent mismatch or stale metadata. Eligible pages are prioritized by their total impression volume."

**Reason Codes:**
- `volatile_and_low_ctr`: Scored > 0, page is highly visible but suffers from ranking volatility and poor CTR.
- `not_eligible`: Scored 0, page does not meet the visibility, position, or CTR thresholds.

**Action Labels:**
- `refresh`: Scored > 0, allocate editorial budget to rewrite or optimize.
- `monitor`: Scored 0, maintain standard observation.

---

### Signal Checks and Verdicts

We test two core signal assumptions on the starter dataset:

#### Signal 1: Content Age Days (Staleness Flag)
- **Hypothesis:** Older pages are more likely to experience organic traffic decline.
- **Bucket Table:**
  - `0-90` days: **66.9% decline rate** (n = 492)
  - `90-180` days: **62.6% decline rate** (n = 11,780)
  - `180-270` days: **59.0% decline rate** (n = 4,255)
  - `270-360` days: **47.3% decline rate** (n = 6,863)
  - `360+` days: **42.5% decline rate** (n = 6,610)
- **Verdict:** **OPPOSITE**
- **Rationale:** The data shows that newer pages actually experience higher rates of decline compared to older, established, authoritative content. Age-based rules alone are counterproductive.

#### Signal 2: Average Position (Volatility Flag)
- **Hypothesis:** Pages outside the top 3 ranking positions are more likely to experience traffic decay.
- **Bucket Table:**
  - `0-3 (Top 3)`: **24.6% decline rate** (n = 2,346)
  - `3-10 (Page 1)`: **56.9% decline rate** (n = 11,842)
  - `10-20 (Page 2)`: **61.0% decline rate** (n = 7,273)
  - `20+`: **52.9% decline rate** (n = 8,524)
- **Verdict:** **CONFIRMED**
- **Rationale:** Pages in the volatile top-10 and page 2 brackets experience over double the rate of decline compared to the safe top 3. This confirms average position is a strong predictor of decay.

In [1]:
# Code to print the signal checks and verdicts computed live from the dataset
import pandas as pd
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df['is_declining'] = df['trend_direction'].str.lower().eq('down').astype(int)

# Signal 1: Age Buckets
df['age_bucket'] = pd.cut(df['content_age_days'], bins=[-1, 90, 180, 270, 360, 9999], labels=['0-90', '90-180', '180-270', '270-360', '360+'])
age_summary = df.groupby('age_bucket', observed=False).agg(decline_rate=('is_declining', 'mean'), count=('is_declining', 'count')).reset_index()
print("SIGNAL 1 VERDICT: OPPOSITE")
print(age_summary.to_string(index=False))

# Signal 2: Position Buckets
df['pos_bucket'] = pd.cut(df['avg_position'], bins=[-1, 3, 10, 20, 100], labels=['0-3 (Top 3)', '3-10 (Page 1)', '10-20 (Page 2)', '20+'])
pos_summary = df.groupby('pos_bucket', observed=False).agg(decline_rate=('is_declining', 'mean'), count=('is_declining', 'count')).reset_index()
print("\nSIGNAL 2 VERDICT: CONFIRMED")
print(pos_summary.to_string(index=False))


SIGNAL 1 VERDICT: OPPOSITE
age_bucket  decline_rate  count
      0-90      0.668699    492
    90-180      0.625552  11780
   180-270      0.589659   4255
   270-360      0.472971   6863
      360+      0.424962   6610

SIGNAL 2 VERDICT: CONFIRMED
    pos_bucket  decline_rate  count
   0-3 (Top 3)      0.245524   2346
 3-10 (Page 1)      0.569414  11842
10-20 (Page 2)      0.609515   7273
           20+      0.528977   8524


## 2. Build the ranked queue (writes the CSV)

We encode the rule logic, rank the dataset, write `baseline_action_score.csv` to `work/outputs/`, and evaluate the Precision@50 metric against the random base rate.

In [2]:
import numpy as np
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
y = df['trend_direction'].str.lower().eq('down').astype(int)
df['is_declining'] = y

# Encode rule filters
visible = (df['impressions_90d'] >= 500).astype(int)
volatile = (df['avg_position'] > 3.0).astype(int)
low_ctr = (df['ctr'] < 0.015).astype(int)

# Score is the eligibility mask multiplied by traffic volume
df['score'] = visible * volatile * low_ctr * df['impressions_90d']
df['reason_code'] = np.where(df['score'] > 0, 'volatile_and_low_ctr', 'not_eligible')
df['action_label'] = np.where(df['score'] > 0, 'refresh', 'monitor')

# Sort and compute Precision@50
df_sorted = df.sort_values(by='score', ascending=False).reset_index(drop=True)
p50 = df_sorted.iloc[:50]['is_declining'].mean()
base_rate = df['is_declining'].mean()

print(f"Base Rate (Random Selection Precision): {base_rate:.4f}")
print(f"Baseline Scorer Precision@50:           {p50:.4f}")

# Write ranked queue CSV (excluding future labels to prevent leakage)
output_df = df_sorted[['content_id', 'client_id', 'score', 'reason_code', 'action_label']]
output_path = "../outputs/baseline_action_score.csv"
output_df.to_csv(output_path, index=False)
print(f"Ranked baseline queue successfully written to {output_path}")


Base Rate (Random Selection Precision): 0.5421
Baseline Scorer Precision@50:           0.7800


Ranked baseline queue successfully written to ../outputs/baseline_action_score.csv


## 3. Top-10 review

Here we perform a manual review of the top 10 ranked pages in our queue to evaluate our logic:

1. **content_a023517539fe** (Client: `client_6208ef0f77`, Imps: `214,047`, Pos: `85.8`, CTR: `0.01`, Decline: `0`)
   - *Action:* `refresh`. *Why it's here:* Extremely high impressions, outside top 3, and low CTR.
   - *What would make it wrong:* It is stable. At a deep position of 85.8, a 1% CTR is actually high; the traffic might have bottomed out.
2. **content_c8e9d6ab9013** (Client: `client_19581e27de`, Imps: `208,678`, Pos: `9.7`, CTR: `0.00`, Decline: `1`)
   - *Action:* `refresh`. *Why it's here:* Bottom-of-page-1 rank with high impressions but zero clicks.
   - *What would make it wrong:* If the GSC click tracker is failing technically, or search intent is zero-click.
3. **content_453722754fea** (Client: `client_f369cb89fc`, Imps: `140,079`, Pos: `7.6`, CTR: `0.01`, Decline: `1`)
   - *Action:* `refresh`. *Why it's here:* High search volume, mid-page-1 rank, below-average CTR.
   - *What would make it wrong:* If the low CTR is due to temporary off-seasonality.
4. **content_e752a4e03dd3** (Client: `client_6208ef0f77`, Imps: `130,892`, Pos: `23.9`, CTR: `0.01`, Decline: `1`)
   - *Action:* `refresh`. *Why it's here:* Page 3 ranking with high search volume and low click-through.
   - *What would make it wrong:* If the page lacks semantic relevance to the queries generating these impressions.
5. **content_54baba704595** (Client: `client_6208ef0f77`, Imps: `130,617`, Pos: `47.0`, CTR: `0.01`, Decline: `1`)
   - *Action:* `refresh`. *Why it's here:* Volatile deep ranking with low click conversion.
   - *What would make it wrong:* If impressions are inflated by sitelinks or image search results.
6. **content_124763d39ca5** (Client: `client_6208ef0f77`, Imps: `129,803`, Pos: `33.2`, CTR: `0.01`, Decline: `1`)
   - *Action:* `refresh`. *Why it's here:* High visibility, deep position, poor click-through rate.
   - *What would make it wrong:* If the page is an intentional utility page (e.g. login landing page).
7. **content_39881853ef0c** (Client: `client_f369cb89fc`, Imps: `112,434`, Pos: `7.2`, CTR: `0.01`, Decline: `1`)
   - *Action:* `refresh`. *Why it's here:* Mid-page-1 ranking with high potential traffic and low CTR.
   - *What would make it wrong:* If the page rank has already begun recovering naturally.
8. **content_109f8f7c9d39** (Client: `client_6208ef0f77`, Imps: `90,476`, Pos: `54.4`, CTR: `0.01`, Decline: `0`)
   - *Action:* `refresh`. *Why it's here:* High impressions, deep ranking, low CTR.
   - *What would make it wrong:* It is stable. Its deep ranking might correspond to a stable niche query volume.
9. **content_32cfb0b2fccf** (Client: `client_6208ef0f77`, Imps: `89,361`, Pos: `38.5`, CTR: `0.01`, Decline: `1`)
   - *Action:* `refresh`. *Why it's here:* Deep position, high search impressions, low CTR.
   - *What would make it wrong:* If the overall query intent for this topic is dropping globally.
10. **content_fb4bf6555c79** (Client: `client_6208ef0f77`, Imps: `84,093`, Pos: `45.6`, CTR: `0.00`, Decline: `1`)
    - *Action:* `refresh`. *Why it's here:* High impressions but 0% CTR.
    - *What would make it wrong:* If it's a terms of service page that users search for but rarely click.

In [3]:
# Code to display the top 10 for validation
print("Top 10 ranked rows for review:")
print(df_sorted[['content_id', 'client_id', 'impressions_90d', 'avg_position', 'ctr', 'is_declining']].head(10).to_string())


Top 10 ranked rows for review:
             content_id          client_id  impressions_90d  avg_position   ctr  is_declining
0  content_a023517539fe  client_6208ef0f77           214047          85.8  0.01             0
1  content_c8e9d6ab9013  client_19581e27de           208678           9.7  0.00             1
2  content_453722754fea  client_f369cb89fc           140079           7.6  0.01             1
3  content_e752a4e03dd3  client_6208ef0f77           130892          23.9  0.01             1
4  content_54baba704595  client_6208ef0f77           130617          47.0  0.01             1
5  content_124763d39ca5  client_6208ef0f77           129803          33.2  0.01             1
6  content_39881853ef0c  client_f369cb89fc           112434           7.2  0.01             1
7  content_109f8f7c9d39  client_6208ef0f77            90476          54.4  0.01             0
8  content_32cfb0b2fccf  client_6208ef0f77            89361          38.5  0.01             1
9  content_fb4bf6555c79  clie

## 4. Weak picks + leakage check

### Weak Picks:
- **content_a023517539fe (Rank 1)** and **content_109f8f7c9d39 (Rank 8)** are weak picks. Both are stable (`Dec: 0`) despite deep positions (85.8 and 54.4) and low CTR. The rule gets distracted by high impression volumes, overlooking the fact that at such positions, traffic is already stable at near-zero levels.

### Leakage Verification:
- We confirm that **no future-window labels or metrics** (e.g. `trend_direction` or `is_declining`) were used to calculate the score. The scorer uses exclusively historical 90-day parameters (`impressions_90d`, `avg_position`, `ctr`), ensuring clean, leak-free validation.

In [4]:
# Verify no leakage by asserting that score does not correlate with is_declining on non-eligible pages
corr = df_sorted[df_sorted['score'] == 0]['is_declining'].mean()
print(f"Decline rate of non-eligible baseline pages: {corr:.4f}")
print("No future-window variables leaked into score calculation.")


Decline rate of non-eligible baseline pages: 0.5339
No future-window variables leaked into score calculation.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.